# Vector RAG Pipeline on the OrchestrAI Synthetic Enterprise Corpus

This notebook demonstrates an advanced Retrieval-Augmented Generation (RAG) pipeline built on the same OrchestrAI synthetic enterprise corpus as the basic notebook.

Compared with the basic notebook, this version adds:
- richer document processing through **Docling**
- tokenizer-aware chunking with **HybridChunker**
- an additional safety split before embedding
- a real **Milvus** vector database backend
- explicit source metadata normalization for cleaner retrieval and citations

## What is different from the basic notebook?

### Basic pipeline
- direct DOCX conversion
- simple splitting
- in-memory document store
- simpler indexing flow

### Advanced pipeline
- richer conversion and chunking
- persistent vector storage in Milvus
- more robust source metadata handling
- a more realistic enterprise-style retrieval pipeline

## Environment requirements

This notebook assumes:
- Python **3.11**
- Ollama is running locally at `http://localhost:11434`
- the Ollama embedding model `mxbai-embed-large` is available
- the Ollama chat model `llama3.1` is available
- a **Milvus server** is running at `http://localhost:19530`
- the OrchestrAI corpus is stored under `./dummy_data`

Run the optional install cell once if packages are missing.

## Imports and runtime checks

In [1]:
import sys
import warnings
import time
from pathlib import Path
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

warnings.filterwarnings("ignore")

print("Python:", sys.executable)
print("Ollama tags status:", requests.get("http://localhost:11434/api/tags").status_code)

import importlib
import results_logger
importlib.reload(results_logger)
from results_logger import save_result_row, clear_results_for_implementation
clear_results_for_implementation("vectordb")

from haystack import Pipeline
from haystack.dataclasses import ChatMessage
from haystack.components.builders import ChatPromptBuilder
from haystack.components.writers import DocumentWriter
from haystack.components.preprocessors import DocumentSplitter, DocumentCleaner

from milvus_haystack import MilvusDocumentStore
from milvus_haystack.milvus_embedding_retriever import MilvusEmbeddingRetriever

from haystack_integrations.components.embedders.ollama import OllamaDocumentEmbedder, OllamaTextEmbedder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

from haystack_integrations.components.converters.docling import DoclingConverter
from docling.chunking import HybridChunker

print("Imports work")

Python: /Users/nikos/Thesis/.venv/bin/python
Ollama tags status: 200
Cleared existing rows for implementation='vectordb'
Imports work


## Connect to Milvus

This notebook uses a Milvus server connection rather than Milvus Lite.  
This keeps the advanced notebook aligned with a real vector database deployment.

In [2]:
document_store = MilvusDocumentStore(
    connection_args={"uri": "http://localhost:19530"},
    drop_old=True,
)

print("Milvus ready")

Milvus ready


## Load the OrchestrAI dataset

The corpus is organized by department under `./dummy_data`, and all `.docx` files are loaded recursively.

In [ ]:
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "dummy_data"
FILES = [str(f.resolve()) for f in DOCUMENTS_DIR.rglob("*.docx")]

print("Files found:", len(FILES))
for f in FILES[:10]:
    print("-", Path(f).name)

Files found: 54
- VAL-SALES-003_Campaign_Approval_Workflow_refreshed.docx
- VAL-SALES-004_CRM_Field_Governance_Note_refreshed.docx
- VAL-SALES-005_Discount_Exception_Policy_refreshed.docx
- VAL-SALES-001_Lead_Routing_Rules_refreshed.docx
- VAL-SALES-002_Webinar_Launch_Brief_refreshed.docx
- VAL-SEC-003_Patch_Compliance_Reminder.docx
- VAL-SEC-004_Data_Retention_Notice.docx
- VAL-SEC-005_Third_Party_Access_Approval_Standard.docx
- VAL-SEC-002_Privileged_Access_Review_Memo.docx
- VAL-SEC-001_Phishing_Alert_Bulletin.docx


## Advanced indexing strategy

To make the advanced pipeline reliable, the indexing flow is:

1. Convert each file with `DoclingConverter`
2. Attach explicit source metadata (`file_name`, `file_path`, `department`)
3. Apply an additional `DocumentSplitter` before embedding
4. Remove metadata fields that Milvus cannot store cleanly
5. Generate embeddings with Ollama
6. Write the embedded chunks to Milvus

This file-by-file indexing approach fixes the earlier `unknown source` problem during retrieval.

In [4]:
OLLAMA_EMBED_MODEL = "mxbai-embed-large"
HF_TOKENIZER_ID = "mixedbread-ai/mxbai-embed-large-v1"

converter = DoclingConverter(
    chunker=HybridChunker(
        tokenizer=HF_TOKENIZER_ID,
        max_tokens=300,
    )
)

doc_embedder = OllamaDocumentEmbedder(
    model=OLLAMA_EMBED_MODEL,
    url="http://localhost:11434"
)

writer = DocumentWriter(document_store=document_store)

def normalize_meta(meta):
    cleaned = {}
    for key, value in (meta or {}).items():
        if key == "_split_overlap":
            continue
        if isinstance(value, (str, int, float, bool)) or value is None:
            cleaned[key] = value
        else:
            cleaned[key] = str(value)
    return cleaned

indexing_start = time.perf_counter()

for file_path in FILES:
    file_path = Path(file_path)

    result = converter.run(sources=[file_path])
    converted_docs = result["documents"]

    for doc in converted_docs:
        if doc.meta is None:
            doc.meta = {}
        doc.meta["file_name"] = file_path.name
        doc.meta["file_path"] = str(file_path.resolve())
        doc.meta["department"] = file_path.parent.name

    cleaner = DocumentCleaner()
    cleaned_docs = cleaner.run(documents=converted_docs)["documents"]

    for doc in cleaned_docs:
        if doc.meta is None:
            doc.meta = {}
        doc.meta["file_name"] = file_path.name
        doc.meta["file_path"] = str(file_path.resolve())
        doc.meta["department"] = file_path.parent.name

    splitter = DocumentSplitter(
        split_by="word",
        split_length=80,
        split_overlap=20
    )

    split_docs = splitter.run(documents=cleaned_docs)["documents"]

    for doc in split_docs:
        if doc.meta is None:
            doc.meta = {}
        doc.meta["file_name"] = file_path.name
        doc.meta["file_path"] = str(file_path.resolve())
        doc.meta["department"] = file_path.parent.name
        doc.meta = normalize_meta(doc.meta)

    docs_with_embeddings = doc_embedder.run(split_docs)
    writer.run(documents=docs_with_embeddings["documents"])

indexing_time_s = round(time.perf_counter() - indexing_start, 4)

print("Indexing time (s):", indexing_time_s)
print(f"Indexed {len(FILES)} source files into Milvus.")

Calculating embeddings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.45it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (589 > 512). Running this sequence through the model will result in indexing errors
Calculating embeddings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.32s/it]

Indexing time (s): 32.2486
Indexed 54 source files into Milvus.


## Build the vector RAG pipeline

The answer-generation logic is aligned with the basic notebook:
- use only the retrieved context
- do not invent missing information
- cite source filenames when possible

In [5]:
template = [
    ChatMessage.from_user(
        """
Respond to the User Query using only the provided Context.

General Guidelines:
- Use only the provided context.
- If the answer is not found in the context, say so clearly.
- Do not make up facts, document sections, pages, or citations.
- If the answer comes from multiple documents, cite all relevant documents.
- Respond in the same language as the user’s query.
- Do not use emojis.
- Be professional and concise.
- Do not write a conclusion or follow-up unless the user asks for it.
- If the context contains a direct policy rule, exception, SLA, or escalation instruction, state it directly and do not hedge.
- Prefer the most specific procedural source over broader handbook summaries when both are present.
- If the retrieved context does not explicitly contain the rule, exception, owner, or next step needed to answer the question, say that the retrieved context is insufficient instead of inferring or guessing.
- Do not mention system names, tools, workflows, or teams unless they are explicitly present in the retrieved context.

Citation Rules:
- For every important claim, cite the source document filename.
- If available in the retrieved context, also mention the document ID or department.
- Do not mention source chapter, section, or page unless they are explicitly present in the retrieved context.

Context:
{% for document in documents %}
[Source: {{ document.meta.get("file_name", document.meta.get("file_path", "unknown source")) }}]
{{ document.content }}

{% endfor %}

Question: {{question}}
Answer:
"""
    )
]

text_embedder = OllamaTextEmbedder(
    model=OLLAMA_EMBED_MODEL,
    url="http://localhost:11434"
)

retriever = MilvusEmbeddingRetriever(
    document_store=document_store,
    top_k=10
)

chat_generator = OllamaChatGenerator(
    model="llama3.1",
    url="http://localhost:11434",
    generation_kwargs={
        "temperature": 0.0,
        "top_p": 1.0,
        "seed": 42,
    }
)

prompt_builder = ChatPromptBuilder(
    template=template,
    required_variables=["question", "documents"]
)

advanced_rag_pipeline = Pipeline()
advanced_rag_pipeline.add_component("text_embedder", text_embedder)
advanced_rag_pipeline.add_component("retriever", retriever)
advanced_rag_pipeline.add_component("prompt_builder", prompt_builder)
advanced_rag_pipeline.add_component("llm", chat_generator)

advanced_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
advanced_rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
advanced_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

print("Vector RAG pipeline ready")

Vector RAG pipeline ready


## Helper function for presentation queries

The function below:
- embeds the question
- retrieves the top document chunks
- prints the retrieved sources and snippets
- runs the full vector RAG pipeline
- prints the final answer

In [6]:
def run_vectordb_query(question):
    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("=" * 80)

    retrieval_start = time.perf_counter()

    q_emb = text_embedder.run(text=question)["embedding"]
    retr_out = retriever.run(query_embedding=q_emb, top_k=10)
    top_docs = retr_out["documents"]

    retrieval_time_s = round(time.perf_counter() - retrieval_start, 4)

    print(f"\nRETRIEVED DOCUMENTS: {len(top_docs)}")
    print(f"RETRIEVAL TIME (s): {retrieval_time_s}\n")

    for i, d in enumerate(top_docs, 1):
        source = d.meta.get("file_name", d.meta.get("file_path", "unknown source"))
        department = d.meta.get("department", "unknown department")
        snippet = (d.content or "")[:400].replace("\n", " ")
        print(f"[{i}] Source: {source} | Department: {department}")
        print(f"    {snippet}...")
        print()

    generation_start = time.perf_counter()

    result = advanced_rag_pipeline.run({
        "text_embedder": {"text": question},
        "retriever": {"top_k": 10},
        "prompt_builder": {"question": question},
    })

    answer = result["llm"]["replies"][0].text
    generation_time_s = round(time.perf_counter() - generation_start, 4)

    print("=" * 80)
    print("FINAL ANSWER:")
    print("=" * 80)
    print(answer)
    print(f"\nGENERATION TIME (s): {generation_time_s}")

    return top_docs, answer, retrieval_time_s, generation_time_s

# Saving results in CSV

In [7]:
def run_and_save_vectordb(question, scenario_id):
    top_docs, answer, retrieval_time_s, generation_time_s = run_vectordb_query(question)

    save_result_row(
        implementation="vectordb",
        scenario_id=scenario_id,
        question=question,
        generated_answer=answer,
        top_docs=top_docs,
        retriever_top_k=10,
        reranker_top_k=None,
        indexing_time_s=indexing_time_s,
        retrieval_time_s=retrieval_time_s,
        generation_time_s=generation_time_s,
    )

    return top_docs, answer

### Clear vectordb rows

In [8]:
clear_results_for_implementation("vectordb")

Cleared existing rows for implementation='vectordb'


PosixPath('rag_results.csv')

### Scenario 1 — Basic fact retrieval
This is the clean baseline. It shows whether the system can retrieve one exact operational fact correctly.
- Single-document, single-fact retrieval.
- Tests basic embedding-based retrieval accuracy.
- Expected answer should come from the Customer Success / Support materials.

In [9]:
top_docs_vectordb_s1, answer_vectordb_s1 = run_and_save_vectordb(
    'What is the first-response SLA for a Sev-1 support incident?',
    scenario_id="v1",
)

QUESTION:
What is the first-response SLA for a Sev-1 support incident?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.1072

[1] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    Customer Success and Support Handbook Specialised Q&A **Q: Who owns customer communication during a Sev-1 incident?** **A:** The assigned support lead owns the operational updates, but the Customer Success Manager owns stakeholder alignment and executive communication. If the issue touches release risk or hardware failure, Engineering or Logistics may join the bridge. **Q: What is the target first...

[2] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    Customer Success and Support Handbook Service targets and escalation guide Sev-1 first response, **Target** = 15 min. Sev-1 first response, **Reviewed by** = Support manager. Sev-1 first response, **Notes** = Clock starts when the case is correctly 

### Scenario 2 — Procedural retrieval
This scenario is more complex because the answer is not a single fact, but a short operational process.
- Procedure and checklist retrieval.
- Tests whether the system can retrieve and summarize structured steps clearly.
- Expected answer should come mainly from the HR / People Operations materials.

In [10]:
top_docs_vectordb_s2, answer_vectordb_s2 = run_and_save_vectordb(
    'What steps must be completed before a new employee is considered ready for day one at OrchestrAI?',
    scenario_id="v2",
)

QUESTION:
What steps must be completed before a new employee is considered ready for day one at OrchestrAI?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.1147

[1] Source: VAL-HR-001_Day_One_Onboarding_Checklist.docx | Department: HR
    Day-One Onboarding Checklist **Document ID:** VAL-HR-001 **Owner:** HR & People Operations **Confidentiality:** Internal Use Only Purpose: This checklist standardizes the minimum tasks that must be complete before an OrchestrAI employee can be considered ready for day-one onboarding. **Department**, **Value** = HR & People Operations. **Effective date**, **Value** = 2026-02-24. **Applies to**, **V...

[2] Source: TEAMQA-HR-001_OrchestrAI_HR_and_People_Operations_Handbook.docx | Department: Q&A Hanbooks
    HR and People Operations Handbook Specialised Q&A **Q: What does HR and People Operations own at OrchestrAI?** **A:** The team owns employee record quality, onboarding coordination, policy communication, leave administration, manager guidance, workp

### Scenario 3 — Multi-team operational reasoning
This scenario requires combining information from more than one team to answer a realistic workplace problem.
- Multi-document, cross-team retrieval.
- Tests whether the system can synthesize HR, IT, and Security responsibilities.
- Expected answer should identify both the involved teams and the next actions.

In [11]:
top_docs_vectordb_s3, answer_vectordb_s3 = run_and_save_vectordb(
    'A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?',
    scenario_id="v3",
)

QUESTION:
A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.0917

[1] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    Common Questions **Q: How far in advance should a laptop be requested?** A: Standard new-hire laptop requests should be present in the hiring workflow at least seven business days before the employee start date. International shipments, executive hires, and custom software bundles may require ten to fifteen business days. **Q: Which team starts the request?** A: The request begins with the hiring ...

[2] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    **Scope note:** This document covers standard Windows and macOS employee laptops. Warehouse handhelds, kiosk terminals, and engineering lab devices follow separate fulfillment rules. **Provisioning Timeline Snapshot** Requ

### Scenario 4 — Policy + exception handling
This scenario checks whether the system can distinguish the normal rule from the emergency exception path.
- Policy retrieval with exception reasoning.
- Tests whether the system can identify when a standard workflow may be bypassed.
- Expected answer should combine Finance / Procurement rules with operational urgency.

In [12]:
top_docs_vectordb_s4, answer_vectordb_s4 = run_and_save_vectordb(
    'Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?',
    scenario_id="v4",
)

QUESTION:
Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.0932

[1] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    exceptions such as extra memory or licensed software. **Q: Can employees choose any laptop model they want?** A: No. OrchestrAI uses a standard device catalog to keep support, security baselines, and spare stock manageable. Exceptions are limited to approved engineering, executive, accessibility, or regional procurement cases. **Q: What software is installed by default?** A: Every standard laptop ...

[2] Source: TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx | Department: Q&A Hanbooks
    into actual movement of equipment and materials. At OrchestrAI, the logistics function does not only ship standard hardware. It also supports replacement gateway appliances, calibrated barcode scanners, installation spare

### Scenario 5 — Cross-functional enterprise reasoning
This is the most complex scenario because it requires cross-functional reasoning across several business units.
- Multi-hop, multi-document retrieval.
- Tests whether the system can combine support, product, logistics, and commercial risk signals.
- Expected answer should identify the teams involved and propose a coordinated next-step plan.

In [13]:
top_docs_vectordb_s5, answer_vectordb_s5 = run_and_save_vectordb(
    'A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?',
    scenario_id="v5",
)

QUESTION:
A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.0942

[1] Source: TEAMQA-IT-001_OrchestrAI_IT_Workplace_and_Service_Desk_Handbook.docx | Department: Q&A Hanbooks
    Team Q&A **Q: What does the IT team own at OrchestrAI?** **A:** IT owns end-user computing, collaboration services, ticket triage, endpoint compliance, warehouse device support, and coordination with Security, Engineering, and vendors for shared systems. **Q: How should users request help?** **A:** All support requests should be logged in AtlasDesk unless there is a declared outage or the user can...

[2] Source: VAL-IT-004_Service_Desk_Priority_Matrix.docx | Department: IT
    scenario** = Service degradation blocking a department, executive user, launch event, or time-bound business process.. **P2 High**